### Implementing an API using PurpleAir

In [7]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
import requests
import os
from dotenv import load_dotenv
from datetime import datetime
import folium

In [8]:
load_dotenv()

API_KEY = os.getenv("PURPLEAIR_READ_KEY")
BASE_URL = "https://api.purpleair.com/v1"

headers = {
    "X-API-Key": API_KEY
}

# --- Get a single sensor's data ---
sensor_index = 131075  # Replace with your target sensor's index

response = requests.get(
    f"{BASE_URL}/sensors/{sensor_index}",
    headers=headers,
    params={"fields": "name ,pm2.5, humidity, temperature"}
)

if response.status_code == 200:
    data = response.json()
    print(data)
else:
    print(f"Error {response.status_code}:", response.text)

{'api_version': 'V1.2.0-1.1.45', 'time_stamp': 1775740652, 'data_time_stamp': 1775740630, 'sensor': {'sensor_index': 131075, 'name': 'Mariners Bluff', 'humidity': 100, 'temperature': 83, 'pm2.5': 14.9}}


In [9]:
#Now responses for KY
response = requests.get(
    "https://api.purpleair.com/v1/sensors",
    headers=headers,
    params={
        "fields": "name, pm2.5, humidity, temperature, latitude, longitude",
        "location_type": 0,
        "nwlng": -89.57,
        "nwlat": 39.15,
        "selng": -81.96,
        "selat": 36.49,
    }
)

data = response.json()
print(f"Found {len(data['data'])} sensors")
print(data)

Found 119 sensors
{'api_version': 'V1.2.0-1.1.45', 'time_stamp': 1775740652, 'data_time_stamp': 1775740631, 'location_type': 0, 'max_age': 604800, 'firmware_default_version': '7.02', 'fields': ['sensor_index', 'name', 'latitude', 'longitude', 'humidity', 'temperature', 'pm2.5'], 'data': [[262243, 'TDEC APC - Kingsport - 262243', 36.538925, -82.52163, 38, 56, 6.8], [263785, 'EWV Whitman 1', 37.79283, -82.030815, 43, 53, 10.7], [263797, 'EWV Logan 2', 37.904606, -82.10735, 26, 60, 9.1], [263809, 'EWV Point Pleasant 3', 38.87221, -82.12882, 28, 66, 6.3], [273763, 'Boone Block', 39.08691, -84.50917, 39, 65, 8.1], [13031, 'Mount Vernon,Ind.', 37.94211, -87.89055, 33, 75, 15.2], [276356, 'Floyd Street Old Louisville', 38.23082, -85.75195, 28, 75, 11.6], [278523, 'EWV Five Mile 1', 37.742935, -82.14651, 59, 51, 7.1], [278531, 'EWV Logan 3', 37.881184, -82.081055, 35, 56, 7.5], [278566, 'EWV Point Pleasant 1', 38.91047, -82.11349, 34, 61, 6.6], [279406, 'Barbourville, KY / Knox County', 36.918

In [10]:
fields = data['fields']
sensors = data['data']

for sensor in sensors:
    sensor_dict = dict(zip(fields, sensor))
    print(f"Name: {sensor_dict['name']}")
    print(f"  PM2.5:       {sensor_dict['pm2.5']}")
    print(f"  Humidity:    {sensor_dict['humidity']}")
    print(f"  Temperature: {sensor_dict['temperature']}")
    print()

Name: TDEC APC - Kingsport - 262243
  PM2.5:       6.8
  Humidity:    38
  Temperature: 56

Name: EWV Whitman 1
  PM2.5:       10.7
  Humidity:    43
  Temperature: 53

Name: EWV Logan 2
  PM2.5:       9.1
  Humidity:    26
  Temperature: 60

Name: EWV Point Pleasant 3
  PM2.5:       6.3
  Humidity:    28
  Temperature: 66

Name: Boone Block
  PM2.5:       8.1
  Humidity:    39
  Temperature: 65

Name: Mount Vernon,Ind.
  PM2.5:       15.2
  Humidity:    33
  Temperature: 75

Name: Floyd Street Old Louisville
  PM2.5:       11.6
  Humidity:    28
  Temperature: 75

Name: EWV Five Mile 1
  PM2.5:       7.1
  Humidity:    59
  Temperature: 51

Name: EWV Logan 3
  PM2.5:       7.5
  Humidity:    35
  Temperature: 56

Name: EWV Point Pleasant 1
  PM2.5:       6.6
  Humidity:    34
  Temperature: 61

Name: Barbourville, KY / Knox County
  PM2.5:       9.0
  Humidity:    44
  Temperature: 64

Name: Cincinnati State
  PM2.5:       16.8
  Humidity:    42
  Temperature: 63

Name: Cincy Air Watc

### The API now gives all sensor readings available in Kentucky, about 119 so far until more people buy/register their PurpleAir sensors. With that, we can now make a map using Folium to show air quality readings across KY.

In [11]:
#Center map on Kentucky
m = folium.Map(location=[37.8, -85.8], zoom_start=7)

for sensor in sensors:
    sensor_dict = dict(zip(fields, sensor))
    
    folium.Marker(
        location=[sensor_dict['latitude'], sensor_dict['longitude']],
        popup=folium.Popup(
            f"<b>{sensor_dict['name']}</b><br>"
            f"PM2.5: {sensor_dict['pm2.5']}<br>"
            f"Humidity: {sensor_dict['humidity']}<br>"
            f"Temperature: {sensor_dict['temperature']}",
            max_width=200
        ),
        tooltip=sensor_dict['name']
    ).add_to(m)

m.save("../End_Visualizations/kentucky_sensors_map.html")
print("Map saved.")

Map saved.


### Now after having made the map, we can make a unique function that finds sensors based on the area's air quality. We put in the numerical threshold we're looking for and the given output will be sensors correlating with what we're searching for.

In [12]:
def get_sensors_by_air_quality(threshold):
    """
    Returns sensors where PM2.5 exceeds the given threshold.
    
    Parameters:
        threshold (float): PM2.5 value to filter by
    
    Returns:
        list of dicts with sensor data
    """
    response = requests.get(
        "https://api.purpleair.com/v1/sensors",
        headers=headers,
        params={
            "fields": "name,pm2.5,humidity,temperature,latitude,longitude",
            "location_type": 0,
            "nwlng": -89.57,
            "nwlat": 39.15,
            "selng": -81.96,
            "selat": 36.49,
        }
    )

    data = response.json()
    fields = data['fields']
    sensors = data['data']

    results = []
    for sensor in sensors:
        sensor_dict = dict(zip(fields, sensor))
        if sensor_dict['pm2.5'] is not None and sensor_dict['pm2.5'] > threshold:
            results.append(sensor_dict)

    print(f"Found {len(results)} sensors above PM2.5 threshold of {threshold}")
    return results

# Example usage — EPA's "Good" air quality limit is 12.0
flagged_sensors = get_sensors_by_air_quality(12.0)
for sensor in flagged_sensors:
    print(f"Name: {sensor['name']}, PM2.5: {sensor['pm2.5']}")

Found 47 sensors above PM2.5 threshold of 12.0
Name: Mount Vernon,Ind., PM2.5: 15.2
Name: Cincinnati State, PM2.5: 16.8
Name: Cincy Air Watch- Zoo, PM2.5: 14.4
Name: Cincy Air Watch- Citylink Center, PM2.5: 15.0
Name: OVSA- Sisters, PM2.5: 12.2
Name: OVSA - Ferdinand Fire Tower, PM2.5: 13.0
Name: OVSA- Enterprise Camp, PM2.5: 13.1
Name: OVSA-Franklin, PM2.5: 16.4
Name: OVSA-West Willow Road, PM2.5: 13.5
Name: OVSA- Garvin Park, PM2.5: 16.4
Name: OVSA/Old Huntingburg Road, PM2.5: 12.6
Name: Mt. Washington, PM2.5: 13.1
Name: Murphysboro, PM2.5: 21.7
Name: RogersAir, PM2.5: 24.9
Name: SR32, PM2.5: 14.6
Name: Santa Claus, Indiana, PM2.5: 12.2
Name: Woods Branch Farm, PM2.5: 15.2
Name: Devou Park, PM2.5: 12.7
Name: Mt Washington, PM2.5: 12.2
Name: home, PM2.5: 12.8
Name: JN-Clifton,OH, PM2.5: 12.1
Name: Cincy Air Watch- Eastside Rec, PM2.5: 15.0
Name: Cincy Air Watch- LeBlond Rec, PM2.5: 16.0
Name: Cincy Air Watch and Madtree- Rockdale Academy, PM2.5: 16.6
Name: Cincy Air Watch &amp; Madtre

##### This API was made with the help of Claude AI in some parts to create a workable map and clarify how the API from PurpleAir was meant to be set up and used as it differs from a basic setup I did in my API weather project that I used for comparison.